# BBH curriculum progress

BBH difficulty changes whenever a run is promoted, so raw loss and accuracy are not comparable across the full training timeline. This notebook instead shows the curriculum frontier, the optimizer steps needed to master each level, and where runs are censored at the end of training.

In [ ]:
from collections import defaultdict
from pathlib import Path
from statistics import median
import sys

import matplotlib
if "ipykernel" in sys.modules:
    matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
import numpy as np

def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "experiments").is_dir() and (candidate / "figures" / "plotting_utils.py").is_file():
            return candidate
    raise FileNotFoundError(f"Could not locate repository root above {start}")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from figures.plotting_utils import (
    ARCHITECTURE_COLORS,
    filter_records,
    grouped,
    load_training_records,
    median_curve,
    set_plot_style,
    unique_values,
)

set_plot_style()
RESULT_ROOT = REPO_ROOT / "results" / "bbh"
FIGURE_DIR = REPO_ROOT / "figures"


In [ ]:
records = load_training_records(RESULT_ROOT)
records = [row for row in records if row.get("level") is not None]

print(f"Loaded {len(records)} BBH evaluation checkpoints from {RESULT_ROOT}")
print("tasks:", unique_values(records, "task"))
print("architectures:", unique_values(records, "architecture"))
print("devices:", unique_values(records, "device"))
print("seeds:", unique_values(records, "seed"))
if not records:
    raise ValueError(f"No BBH curriculum metrics found below {RESULT_ROOT}")


## One task in detail

The plot shows how many successive swaps each architecture can track. Faint lines are individual seeds; the heavier line is their median.

In [ ]:
TASK = "permutation"  # tracking, permutation, pointer_chasing, state_machine
DEVICE = None         # set to "mps", "cuda", or "cpu" only when needed
ARCHITECTURES = list(ARCHITECTURE_COLORS)

selected = filter_records(records, task=TASK, device=DEVICE)
selected = [row for row in selected if row.get("architecture") in ARCHITECTURES]
if not selected:
    raise ValueError(f"No BBH records for TASK={TASK!r}. Available tasks: {unique_values(records, 'task')}")
print(f"Selected {len(selected)} checkpoints from {len({row['run_dir'] for row in selected})} runs")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
display_names = {
    "transformer": "Transformer",
    "memory_attention": "MemoryAttention (multi-pass trained)",
    "memory_add": "MemoryAdd (multi-pass trained)",
}

for architecture in ARCHITECTURES:
    architecture_rows = [row for row in selected if row.get("architecture") == architecture]
    if not architecture_rows:
        continue
    color = ARCHITECTURE_COLORS.get(architecture)
    for (_seed,), seed_rows in grouped(architecture_rows, "seed").items():
        points = sorted((row["step"], row["level"]) for row in seed_rows)
        ax.step(*zip(*points), where="post", color=color, alpha=0.18, linewidth=1)
    steps, levels = median_curve(architecture_rows, "level")
    ax.step(steps, levels, where="post", color=color, linewidth=2.4, label=display_names.get(architecture, architecture))

ax.set(xlabel="Training step", ylabel="Number of swaps", title="S₅ permutation tracking")
ax.set_ylim(bottom=0.5)
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
plt.show()
# fig.savefig(FIGURE_DIR / "s5_permutation_fig.png", dpi=220, bbox_inches="tight")


## All BBH tasks

Each task gets its own axis because a level has task-specific meaning. Compare architectures within a panel; do not interpret equal numerical levels across different tasks as equal difficulty.

In [ ]:
tasks = unique_values(records, "task")
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
axes = axes.ravel()
for ax, task in zip(axes, tasks):
    task_rows = [row for row in records if row.get("task") == task
                 and row.get("architecture") in ARCHITECTURES
                 and (DEVICE is None or row.get("device") == DEVICE)]
    for architecture in ARCHITECTURES:
        architecture_rows = [row for row in task_rows if row.get("architecture") == architecture]
        steps, levels = median_curve(architecture_rows, "level")
        if steps:
            ax.step(steps, levels, where="post", color=ARCHITECTURE_COLORS.get(architecture),
                    linewidth=2, label=architecture)
    ax.set(title=task.replace("_", " ").title(), xlabel="Optimizer step", ylabel="Level")
    ax.set_ylim(bottom=0.5)
for ax in axes[len(tasks):]:
    ax.set_axis_off()
handles, labels = axes[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="lower center", ncol=len(labels), bbox_to_anchor=(0.5, -0.01))
fig.suptitle("BBH curriculum frontiers", y=1.01)
fig.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()
# fig.savefig(FIGURE_DIR / "bbh_all_curricula.png", dpi=220, bbox_inches="tight")


## End-of-budget curriculum coverage

The matrix reports the median highest level reached relative to that task's configured maximum. The annotation is `level / maximum`; color represents experimental coverage, not a claim that levels are comparable across tasks.

In [ ]:
coverage_tasks = unique_values(records, "task")
coverage_architectures = list(ARCHITECTURE_COLORS)
latest_by_run = []
for (_run_dir,), run_rows in grouped(records, "run_dir").items():
    latest_by_run.append(max(run_rows, key=lambda row: row["step"]))

coverage = np.full((len(coverage_tasks), len(coverage_architectures)), np.nan)
annotations = [["—" for _ in coverage_architectures] for _ in coverage_tasks]
for task_index, task in enumerate(coverage_tasks):
    for architecture_index, architecture in enumerate(coverage_architectures):
        rows = [row for row in latest_by_run if row.get("task") == task
                and row.get("architecture") == architecture
                and isinstance(row.get("max_level"), (int, float)) and row["max_level"] > 0]
        if not rows:
            continue
        fractions = [row["level"] / row["max_level"] for row in rows]
        level = median(row["level"] for row in rows)
        maximum = median(row["max_level"] for row in rows)
        coverage[task_index, architecture_index] = median(fractions)
        annotations[task_index][architecture_index] = f"{level:g}/{maximum:g}"

fig, ax = plt.subplots(figsize=(10.5, 4.8))
image = ax.imshow(coverage, vmin=0, vmax=1, cmap="viridis", aspect="auto")
for task_index in range(len(coverage_tasks)):
    for architecture_index in range(len(coverage_architectures)):
        value = coverage[task_index, architecture_index]
        if np.isfinite(value):
            text_color = "white" if value < 0.55 else "black"
            ax.text(architecture_index, task_index, annotations[task_index][architecture_index],
                    ha="center", va="center", color=text_color)
ax.set_xticks(range(len(coverage_architectures)), [name.replace("_", "\n") for name in coverage_architectures])
ax.set_yticks(range(len(coverage_tasks)), [task.replace("_", " ") for task in coverage_tasks])
ax.set_title("Median final level / configured maximum")
fig.colorbar(image, ax=ax, label="Fraction of configured curriculum reached")
fig.tight_layout()
plt.show()
